## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python

#!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# For installing the libraries & downloading models from HF Hub
#!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# to suppress unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

## Question Answering using LLM

#### Downloading and Loading the model

In [4]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [5]:
model_path = hf_hub_download(
    repo_id= model_name_or_path, #Complete the code to mention the repo id
    filename= model_basename #Complete the code to mention the model name
)

mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [6]:
# Runtime is connected to GPU.
llm = Llama(
    model_path=model_path,
    n_ctx=4096, # Increased context window to accommodate longer prompts.
    n_gpu_layers=36,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [7]:
def response(query,max_tokens=1024,temperature=0,top_p=0.95,top_k=50): # Temparature is set to 0 to get "factual" answers.
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [8]:
# Creating variables for the common questions and queries.

common_question_1 = "What are the common symptoms and treatments for pulmonary embolism?"
common_question_2 = "What treatment options are available for managing hypertension?"
common_question_3 = "Can you provide the trade names of medications used for treating hypertension?"
common_question_4 = "What are the first-line options and alternatives for managing rheumatoid arthritis?"
common_question_5 = "What are the diagnostic steps for suspected endocrine disorders?"

query_1 = "What is the protocol for managing sepsis in a critical care unit?"
query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

# Creating a non-medical query to test user prompt compliance.

quality_control_query = "How to  clean a grease stain from a white shirt?"

### Generating responses to common questions in medicine.

Let us now use the LLM to answer common medicine questions.

In [9]:
print(common_question_1+"\n")

print(response(common_question_1))

What are the common symptoms and treatments for pulmonary embolism?



Pulmonary embolism is a serious condition that occurs when a blood clot or other foreign substance travels to the lungs and blocks one or more of the arteries supplying blood to the lungs. This can lead to decreased oxygen supply to the body, which can be life-threatening if not treated promptly.

Common symptoms of pulmonary embolism include:

1. Shortness of breath: This is often the most prominent symptom and may be accompanied by chest pain or discomfort.
2. Rapid heart rate: The body may respond to decreased oxygen supply by increasing heart rate in an attempt to pump more blood to the lungs.
3. Coughing up blood: In some cases, pulmonary embolism can cause coughing up blood or bloody sputum.
4. Chest pain: This may be sharp or dull and may worsen with deep breathing or physical activity.
5. Swelling in the legs or arms: Pulmonary embolism can cause fluid buildup in the legs, ankles, or arms due to decreased bl

In [10]:
print(common_question_2+"\n")
print(response(common_question_2))

What treatment options are available for managing hypertension?



Llama.generate: prefix-match hit




Hypertension, or high blood pressure, is a common condition that can increase the risk of various health problems such as heart disease, stroke, and kidney damage. The good news is that there are several effective treatment options available to help manage hypertension and reduce the risk of complications. Here are some of the most commonly used treatments:

1. Lifestyle modifications: Making lifestyle changes is often the first line of defense against hypertension. This may include eating a healthy diet rich in fruits, vegetables, whole grains, and lean proteins; limiting sodium intake; getting regular physical activity; maintaining a healthy weight; and reducing stress through techniques such as meditation or deep breathing exercises.
2. Medications: If lifestyle modifications alone are not enough to control hypertension, medications may be necessary. There are several classes of drugs used to treat hypertension, including diuretics, beta blockers, ACE inhibitors, calcium channel b

In [11]:
print(common_question_3+"\n")
print(response(common_question_3))

Can you provide the trade names of medications used for treating hypertension?



Llama.generate: prefix-match hit




Yes, there are numerous medications used for treating hypertension (high blood pressure), and their trade names can vary depending on the specific drug class and manufacturer. Here are some common classes of antihypertensive drugs and their trade names:

1. Thiazide diuretics: HCTZ (Hydrochlorothiazide), Diuril (Chlorothiazide), Maxzide (Hydrochlorothiazide and amlodipine)
2. Beta-blockers: Tenormin (Atenolol), Lopressor (Metoprolol), Inderal LA (Propranolol)
3. ACE inhibitors: Zestril (Lisinopril), Monopril (Fosinopril), Lotensin (Benazepril)
4. ARBs (Angiotensin II receptor blockers): Cozaar (Losartan), Diovan (Valsartan), Avapro (Irbesartan)
5. Calcium channel blockers: Procardia XL (Nifedipine), Norvasc (Amlodipine), Diltia XT (Diltiazem)
6. Alpha-blockers: Minipress (Phenylephrine), Cardura (Doxazosin), Hytrin (Terazosin)
7. Diuretics (potassium-sparing): Aldactone (Spironolactone), Microzide (Hydrochlorothiazide and spironolactone)
8. Combination medications: Amturnide (Amlodip

In [12]:
print(common_question_4+"\n")
print(response(common_question_4))

What are the first-line options and alternatives for managing rheumatoid arthritis?



Llama.generate: prefix-match hit




Rheumatoid arthritis (RA) is a chronic inflammatory disease that primarily affects the joints, although extra-articular manifestations can also occur. The primary goal of RA treatment is to achieve remission or low disease activity as soon as possible and maintain it for as long as possible.

First-line options for managing rheumatoid arthritis include:

1. Disease-modifying antirheumatic drugs (DMARDs): These medications work by slowing down the progression of joint damage and reducing inflammation. Commonly used DMARDs include methotrexate, sulfasalazine, hydroxychloroquine, leflunomide, and azathioprine.
2. Biologic DMARDs: These are more targeted therapies that work by inhibiting specific molecules involved in the inflammatory process. Examples include tumor necrosis factor (TNF) inhibitors (etanercept, adalimumab, infliximab, certolizumab pegol, and golimumab), interleukin-1 (IL-1) inhibitors (anakinra and canakinumab), and B-cell depleters (rituximab and abatacept).
3. Corticos

In [13]:
print(common_question_5+"\n")
print(response(common_question_5))

What are the diagnostic steps for suspected endocrine disorders?



Llama.generate: prefix-match hit




1. Obtain a thorough history and perform a physical examination: Endocrine disorders can present with a wide range of symptoms, so it is important to obtain a detailed history from the patient. This should include information about their symptoms, duration, frequency, and any associated signs or symptoms. A physical examination may reveal clues to the underlying disorder, such as growths or lesions, abnormal body hair distribution, or signs of metabolic imbalance.
2. Order appropriate laboratory tests: Depending on the suspected endocrine disorder, various laboratory tests may be ordered to help confirm the diagnosis. These may include measurements of hormone levels (such as insulin, glucagon, cortisol, thyroid hormones, and growth hormone), blood glucose levels, electrolyte levels, and other markers of metabolic function. Imaging studies, such as ultrasounds or CT scans, may also be used to visualize glands or tumors.
3. Consider additional diagnostic tests: Depending on the suspect

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [14]:
llm_query_response_1 = response(query_1)
print(llm_query_response_1)

Llama.generate: prefix-match hit




Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:

1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.
2. ABCs: Ensure airway patency, adequate breathing, and circulatory support. Provide high-flow oxygen via a non-rebreather mask or endotracheal tube if necessary. Initiate intravenous fluids to maintain adequate blood pressure and organ perfusion.
3. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on the suspected source of infection and local microbiology data. Consider obtaining cultures before administering antibiotics if clinically 

* Provides a general, comprehensive overview of sepsis management, covering broad steps like early recognition, ABCs, antibiotics, and hemodynamic support.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [15]:
llm_query_response_2 = response(query_2)
print(llm_query_response_2)

Llama.generate: prefix-match hit




Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:

1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that worsens over time.
2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea.
3. Nausea and vomiting: Vomiting is a common symptom of appendicitis, especially in the early stages.
4. Fever: A fever may be present, particularly if the appendix has ruptured or perforated.
5. Constipation or diarrhea: Some people with appendicitis experience constipation, while others have diarrhea.
6. Abdominal swelling: The abdomen may become swollen and tender to the touch.
7. Inability to pass gas: Passing gas can be difficult due to the pressure on the 

* Lists common symptoms such as abdominal pain, nausea, and fever, and correctly identifies surgery (appendectomy) as the necessary treatment.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [16]:
llm_query_response_3 = response(query_3)
print(llm_query_response_3)

Llama.generate: prefix-match hit




Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.

The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications.

There are several treatments that have been shown to be effective in addressing sudden patchy hair loss:

1. Corticosteroids: These are anti-inflammatory drugs that can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally, depending on the severity of the condition.
2. Minoxidil: This is a medication that has been shown to promote hair growth in some people with alopecia areata. It works by increasing bloo

* Identifies alopecia areata and suggests treatments like corticosteroids, minoxidil, and DHT blockers, along with general causes.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [17]:
llm_query_response_4 = response(query_4)
print(llm_query_response_4)

Llama.generate: prefix-match hit




A person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:

1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.
2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a brain injury, such as pain, swelling, seizures, or infections.
3. Surgery: In some cases, surgery may be necessary to remove blood clots, repair skull fractures, or relieve pressure on the brain.
4. Rehabilitation: Rehabilitation is an essential component of treatment for brain injuries. It may include physical therapy, occupational therapy, speech and language thera

* Gives a general list of treatments like emergency care, medication, surgery, and rehabilitation.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [18]:
llm_query_response_5 = response(query_5)
print(llm_query_response_5)

Llama.generate: prefix-match hit




First and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:

1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.
2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.
3. Immobilize the leg: Use a splint, sling, or other available materials to immobilize the leg and prevent movement. Be sure not to apply too much pressure on the injury site.
4. Provide pain relief: Offer over-the-counter pain medication, such as acetaminophen or ibuprofen, to help manage pain.
5. Seek medical attention: If the fracture is severe or if you suspect that there may be other injuries, seek medical help as soon as possible.

Once you've ensured the person's safety and stability, consi

* Provides standard first aid and general recovery advice (immobilize, pain relief, seek medical attention, rehabilitation).

## Question Answering using LLM with Prompt Engineering

In [19]:
# Creating a system prompt with a role of a helpful medical assistant and instruction to answer only medical related queries and summarize answers.

system_prompt = "You are a helpful medical assistant. You will answer questions related to medicine. You will not answer any question that is not related to medicine. Summarize your answer in less than 500 words"

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [20]:
user_input_1 = system_prompt+"\n"+query_1
llm_sp_query_response_1 = response(user_input_1) # with temperature = 0
print(llm_sp_query_response_1)

Llama.generate: prefix-match hit



Sepsis is a life-threatening condition caused by the body's response to an infection. In a critical care unit, managing sepsis requires a quick and coordinated response from the healthcare team to prevent progression to septic shock or multiple organ failure. Here are the general steps for managing sepsis in a critical care unit:
1. Early recognition and diagnosis: The first step is to recognize the signs and symptoms of sepsis early and initiate treatment promptly. This includes monitoring vital signs, assessing mental status, and looking for signs of infection such as fever, chills, or increased heart rate. Laboratory tests, such as blood cultures and lactate levels, can also help confirm the diagnosis.
2. Fluid resuscitation: Septic shock is a complication of sepsis that can lead to low blood pressure and organ failure. Fluid resuscitation is an essential part of managing sepsis in the critical care unit. The goal is to restore intravascular volume and maintain adequate tissue perf

* The response is similar to LLM-only but likely influenced by the *summarization* constraint and *medical assistant* role, leading to a slightly more structured but still general answer.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [21]:
user_input_2 = system_prompt+"\n"+query_2
llm_sp_query_response_2 = response(user_input_2, temperature=0.1) # with temperature = 0.10
print(llm_sp_query_response_2)

Llama.generate: prefix-match hit



Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure located in the lower right side of the abdomen. The common symptoms of appendicitis include:
1. Abdominal pain: The pain usually starts as a mild discomfort around the navel area and then moves to the lower right side of the abdomen. The pain may be constant or intermittent and worsens with movement, coughing, or sneezing.
2. Loss of appetite: Patients with appendicitis often lose their appetite due to abdominal discomfort and nausea.
3. Nausea and vomiting: Vomiting is a common symptom in appendicitis, and it may occur before or after the onset of abdominal pain.
4. Fever: A low-grade fever (100.4°F or 38°C) is often present in appendicitis.
5. Constipation or diarrhea: Both constipation and diarrhea can occur in appendicitis, depending on the location of the inflammation and the degree of obstruction of the appendix.
Appendicitis cannot be cured via medicine alone. Antibio

* Answer is similar to LLM-only, as it provides general symptoms and confirmes surgery as the necessary treatment, adding that *antibiotics alone wouldn't resolve it*.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [22]:
user_input_3 = system_prompt+"\n"+query_3
llm_sp_query_response_3 = response(user_input_3, temperature=0.25) # with temperature = 0.25
print(llm_sp_query_response_3)

Llama.generate: prefix-match hit




Sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that causes hair loss in small patches on the scalp or other parts of the body. The exact cause of alopecia areata is unknown, but it's believed to be related to a problem with the immune system.

The good news is that there are several effective treatments for addressing sudden patchy hair loss:

1. Corticosteroids: These medications can be applied directly to the affected area or taken orally to reduce inflammation and suppress the immune system response. Topical corticosteroids, such as minoxidil foam or lotion, can be used in combination with oral corticosteroids for better results.
2. Immunotherapy: This treatment involves injecting small doses of certain substances into the affected area to stimulate an immune response and promote hair regrowth. The most common immunotherapies used for alopecia areata are squamous cell carcinoma vaccine (SCCV) and diphenylcyclopropenone (DPCP).
3. Minoxidil: This 

* The answer is more structured, similar to LLM-only, but explicitly lists *possible causes* after *effective treatments*.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [23]:
user_input_4 = system_prompt+"\n"+query_4
llm_sp_query_response_4 = response(user_input_4, temperature=0.4) # with temperature = 0.4
print(llm_sp_query_response_4)

Llama.generate: prefix-match hit




A traumatic brain injury (TBI) can result from various causes such as motor vehicle accidents, sports injuries, falls, or violence. The treatment for TBI depends on the severity and location of the injury. Here are some common treatments:

1. Emergency care: In case of a severe TBI, the first priority is to ensure the person's safety and provide emergency care. This may include controlling bleeding, preventing further injury, and maintaining adequate oxygenation and circulation.
2. Surgery: If there is an intracranial hematoma or other lesion that requires surgical intervention, it should be performed as soon as possible to reduce pressure on the brain and prevent further damage.
3. Medications: Depending on the symptoms, various medications may be prescribed to manage conditions such as seizures, pain, inflammation, or increased intracranial pressure.
4. Rehabilitation: Rehabilitation is an essential component of TBI treatment. It may include physical therapy to improve mobility, oc

* The response has a more organized list of general treatments, focusing on comprehensive care.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [24]:
user_input_5 = system_prompt+"\n"+query_5
llm_sp_query_response_5 = response(user_input_5, temperature=0.5) # with temperature = 0.5
print(llm_sp_query_response_5)

Llama.generate: prefix-match hit




A leg fracture sustained during a hiking trip requires prompt medical attention. Here's a summary of necessary precautions and treatment steps:

1. Assess the severity of the injury: Determine if it is an open or closed fracture, and check for signs of nerve damage or impaired circulation. Open fractures require more immediate medical attention due to the risk of infection.
2. Immobilize the leg: Use a splint, sling, or a makeshift immobilizer made from available materials to prevent further damage and discomfort. Ensure that the person is as comfortable as possible during transport.
3. Call for emergency medical services (EMS) or arrange for transportation to a hospital: If the fracture is severe or if there are other injuries, call for EMS or have someone else make arrangements for transportation to the hospital. Do not attempt to move the person unnecessarily, as this could worsen their injury.
4. Manage pain: Provide pain relief using over-the-counter medications like acetaminoph

* Similar to LLM-only, it provides general steps for immediate care and recovery with some additional considerations like nutrition and hydration.

### Testing if model will answer a non-medical query.

In [25]:
user_input_6 = system_prompt+"\n"+quality_control_query
llm_sp_qc_response = response(user_input_6)
print(quality_control_query+"\n")
print(llm_sp_qc_response)

Llama.generate: prefix-match hit


How to  clean a grease stain from a white shirt?


To effectively remove a grease stain from a white shirt, follow these steps:
1. Act quickly: The sooner you attend to the stain, the easier it will be to remove.
2. Blot the excess grease: Use a clean cloth or paper towel to blot up as much of the excess grease as possible. Be careful not to spread the stain.
3. Apply dish soap: Dab a small amount of dish soap directly onto the stain. Gently rub it in with your fingers, making sure to work it into all areas of the stain.
4. Let it sit: Allow the dish soap to sit on the stain for a few minutes to help break down the grease.
5. Rinse with warm water: Rinse the shirt under warm running water to remove the soap and as much of the grease as possible. Be sure to rinse both sides of the shirt.
6. Wash the shirt: Wash the shirt in the warmest water recommended on the care label, using a heavy-duty detergent. Do not overload the washing machine.
7. Check the stain: Inspect the stain after washi

* Provides a detailed, practical, and accurate step-by-step guide for cleaning a grease stain, as expected from a general-purpose model.

* It should be noted that although the prompt directs to not answer non medical queries, the model still provides a detailed, practical, and accurate answer, showing a slight lack of adherence to the negative constraint.


## Data Preparation for RAG

### Loading the Data

In [26]:
manual_pdf_path = "medical_diagnosis_manual.pdf"

In [27]:
pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [28]:
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [29]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
roberto@amtec.cr
0EY2ZWFUC7
This file is meant for personal use by roberto@amtec.cr only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
roberto@amtec.cr
0EY2ZWFUC7
This file is meant for personal use by roberto@amtec.cr only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...................................................................................................................................................

#### Checking the number of pages

In [30]:
len(manual)

4114

* The manual has 4114 pages.

### Data Chunking

In [31]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1000, # Setting most commonly used chunk size.
    chunk_overlap=100 # Setting most commonly used chunk overlap.
)

In [32]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [33]:
len(document_chunks)

4685

In [34]:
document_chunks[0].page_content

'roberto@amtec.cr\n0EY2ZWFUC7\nThis file is meant for personal use by roberto@amtec.cr only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [35]:
document_chunks[2].page_content

"Table of Contents\n1\nFront    ................................................................................................................................................................................................................\n1\nCover    .......................................................................................................................................................................................................\n2\nFront Matter    ...........................................................................................................................................................................................\n53\n1 - Nutritional Disorders    ...............................................................................................................................................................\n53\nChapter 1. Nutrition: General Considerations    ...........................................................................................

In [36]:
document_chunks[3].page_content

'491\nChapter 44. Foot & Ankle Disorders    .....................................................................................................................................\n502\nChapter 45. Tumors of Bones & Joints    ...............................................................................................................................\n510\n5 - Ear, Nose, Throat & Dental Disorders    ..................................................................................................................\n510\nChapter 46. Approach to the Patient With Ear Problems    ...........................................................................................\n523\nChapter 47. Hearing Loss    .........................................................................................................................................................\n535\nChapter 48. Inner Ear Disorders    ...................................................................................................

### Embedding

In [37]:
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [38]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [39]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  384


True

In [40]:
embedding_1,embedding_2

([-0.07455842941999435,
  0.0914825052022934,
  -0.08927078545093536,
  -0.012111259624361992,
  0.06440259516239166,
  0.02896791510283947,
  -0.020714644342660904,
  0.08247439563274384,
  0.04008309543132782,
  0.02838187664747238,
  0.08396773040294647,
  0.046048179268836975,
  0.03379371762275696,
  -0.022817598655819893,
  -0.0750894621014595,
  -0.040113795548677444,
  -0.08240155875682831,
  0.0066926367580890656,
  -0.013369264081120491,
  0.04779652878642082,
  -0.02642734907567501,
  0.016969870775938034,
  0.005332064349204302,
  0.01177381630986929,
  0.02546234056353569,
  0.010443813167512417,
  -0.028520071879029274,
  0.07009293884038925,
  -0.05668044462800026,
  -0.06140447407960892,
  -0.01901652105152607,
  0.008642353117465973,
  0.09434433281421661,
  0.009052395820617676,
  0.06406552344560623,
  0.0071924226358532906,
  0.019878510385751724,
  -0.0904499888420105,
  -0.008894519880414009,
  -0.023051980882883072,
  -0.03129636496305466,
  0.0035739801824092865

### Vector Database

In [41]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [42]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [43]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

In [44]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [45]:
vectorstore.similarity_search("What is sepsis?",k=3)

[Document(metadata={'format': 'PDF 1.7', 'file_path': 'medical_diagnosis_manual.pdf', 'page': 2453, 'trapped': '', 'creationDate': 'D:20120615054440Z', 'total_pages': 4114, 'moddate': '2026-03-19T16:43:13+00:00', 'author': '', 'subject': '', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'keywords': '', 'source': 'medical_diagnosis_manual.pdf', 'creationdate': '2012-06-15T05:44:40+00:00', 'modDate': 'D:20260319164313Z', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition'}, page_content='Chapter 227. Sepsis and Septic Shock\nIntroduction\n(See also Ch. 226.)\nSepsis, severe sepsis, and septic shock are inflammatory states resulting from the systemic\nresponse to bacterial infection. In severe sepsis and septic shock, there is critical reduction in\ntissue perfusion. Common causes include gram-negative organisms, staphylococci, and\nmeningococci. Symptoms often begin with shaking chills and include fever, hypotension,\noli

### Retriever

In [46]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3} # Retrieve top 3 similar documents.
)

In [47]:
rel_docs = retriever.get_relevant_documents("What is sepsis?")

In [48]:
model_output = llm(
      "What is sepsis?",
      max_tokens=512,
      temperature=0,
    )

Llama.generate: prefix-match hit


In [49]:
print(model_output['choices'][0]['text'])



Sepsis is a potentially life-threatening condition that arises when the body’s response to an infection, instead of helping, causes damage to its own tissues and organs. Sepsis occurs when the immune system overreacts to an infection, releasing chemicals into the bloodstream to fight it. This response can trigger inflammation throughout the body and cause damage to multiple organ systems, including the lungs, heart, kidneys, and brain.

Sepsis is a serious condition that requires prompt medical attention. It can lead to septic shock, which is a potentially life-threatening complication in which blood pressure drops dramatically and organs may fail. Septic shock is a medical emergency and requires immediate treatment in a hospital setting.

Who is at risk for sepsis?

Anyone can develop sepsis, but certain people are at higher risk than others. These include:

- Older adults
- Young children
- People with weakened immune systems due to illness or medication
- People with chronic condi

* The response is generic and based on the data the model was trained on, rather than the medical manual.

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [50]:
qna_system_message = "You are a medical assistant and answer the questions based only on the provided context. Summarize your answer in less than 500 words."

In [51]:
qna_user_message_template = """Context: {context}
Question: {question}"""

### Response Function

In [52]:
def generate_rag_response(user_input,k=3,max_tokens=1024,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [53]:
rag_initial_response_1 = generate_rag_response(query_1,k=1)
print(rag_initial_response_1)

Llama.generate: prefix-match hit


Answer: Sepsis is a life-threatening condition that requires prompt recognition and intervention in a critical care unit. The protocol for managing sepsis includes early recognition of signs and symptoms, such as fever, tachycardia, respiratory distress, and altered mental status. Once suspected, sepsis should be confirmed with laboratory tests, including blood cultures and lactate levels.
Initial management involves providing adequate fluid resuscitation to maintain adequate tissue perfusion, administering broad-spectrum antibiotics, and addressing any underlying source of infection. Close monitoring of vital signs, urine output, and oxygenation is essential.
In cases of severe sepsis or septic shock, additional interventions may be necessary, such as vasopressors to maintain blood pressure, inotropes to improve cardiac output, and corticosteroids to reduce inflammation. Renal replacement therapy may also be considered for patients with acute kidney injury.
Throughout the management o

* Offers a more specific and structured protocol, incorporating medical scoring systems (SOFA score), specific types of vasopressors, and advanced treatment modalities (CRRT). Also acknowledges that protocols might vary between institutions, reflecting a deeper, context-aware understanding from the medical manual.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [54]:
rag_initial_response_2 = generate_rag_response(query_2,k=1)
print(rag_initial_response_2)

Llama.generate: prefix-match hit


Answer: The common symptoms of appendicitis include epigastric or periumbilical pain followed by nausea, vomiting, anorexia, and a shift in pain to the right lower quadrant. Pain increases with cough and motion, and there may be direct and rebound tenderness at McBurney's point. Additional signs include Rovsing sign, psoas sign, obturator sign, and low-grade fever. However, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur. Appendicitis cannot be cured via medicine alone; surgery is required to remove the inflamed appendix before it ruptures and causes complications such as peritonitis or an abscess. The surgical procedure for treating appendicitis is typically a laparotomy, but laparoscopy can also be used for diagnosis and treatment. Without surgery or antibiotics, mortality is over 50%. With early surgery, the mortality rate is less than 1%, and convalescence is normally rapid and complete.


* Provides significantly more detail than only LLM, including specific pain characteristics (epigastric/periumbilical shifting to right lower quadrant), precise medical signs (Rovsing, psoas, obturator, McBurney's point), and notes that classic findings are often absent. It details the medical consequences of non-treatment and the specific surgical procedures (laparotomy/laparoscopy).

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [55]:
rag_initial_response_3 = generate_rag_response(query_3,k=1)
print(rag_initial_response_3)

Llama.generate: prefix-match hit


Answer: Alopecia areata is a common condition characterized by sudden, patchy hair loss with no obvious skin or systemic disorder. The scalp and beard are most frequently affected, but any hairy area may be involved. In severe cases, all body hair (alopecia universalis) can be lost. Alopecia areata is believed to be an autoimmune disorder affecting genetically susceptible individuals exposed to unclear environmental triggers.
Effective treatments for alopecia areata include:
1. Topical corticosteroids: These medications help reduce inflammation and promote hair regrowth. They can be applied directly to the affected area or used in combination with intralesional steroid injections.
2. Minoxidil (Rogaine): This medication is available over-the-counter and can stimulate hair growth when applied topically to the scalp twice daily.
3. Immunotherapy: Diphencyprone or squaric acid dibutylester can be used for topical immunotherapy, which involves applying a small amount of the substance to th

* Expands on alopecia areata with specific corticosteroid types (topical, intralesional, systemic) and immunotherapies. Crucially, it also covers treatments for other hair loss conditions mentioned in the manual, such as traction alopecia, tinea capitis, and chemotherapy-induced hair loss, demonstrating a broader context-specific knowledge.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [56]:
rag_initial_response_4 = generate_rag_response(query_4,k=1)
print(rag_initial_response_4)

Llama.generate: prefix-match hit


Answer: Traumatic brain injury (TBI) occurs when there is damage to the brain due to external physical force. The extent and duration of impairment vary widely depending on the severity of the injury. In some cases, individuals may experience a persistent vegetative state, where they show signs of wakefulness but lack consciousness and cannot communicate effectively. However, autonomic and motor reflexes are preserved, and sleep-wake cycles are normal. Unfortunately, few patients recover normal neurological function when a persistent vegetative state lasts for three months after injury, and almost none recover after six months. Treatment for TBI focuses on managing symptoms, preventing complications, and promoting recovery. This may include:
1. Supportive care: Providing adequate nutrition, hydration, and maintaining airway patency to ensure the individual's overall health and well-being.
2. Medications: Administering medications to manage symptoms such as pain, seizures, or increased 

* Focuses on specific injury outcomes like persistent vegetative state and brain death, detailing prognosis and the emphasis on supportive care and rehabilitation in such severe cases. It also mentions specific neurological syndromes (agnosia, anosognosia), reflecting information likely found in the specialized manual.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [57]:
rag_initial_response_5 = generate_rag_response(query_5,k=1)
print(rag_initial_response_5)

Llama.generate: prefix-match hit


Answer: For a person who has fractured their leg during a hiking trip, the first priority is to assess the severity of the injury and ensure their safety. If there are signs of hemorrhagic shock or life-threatening injuries, immediate medical attention should be sought. For suspected arterial injuries, arteriography may be necessary.
For most fractures, initial treatment includes immobilization to prevent further injury and decrease pain. Splinting with a nonrigid or noncircumferential device is used for unstable injuries and long-bone fractures to prevent fat embolism. Pain is typically treated with opioids. Definitive treatment often involves reduction, which may require analgesia or sedation. Closed reduction without skin incision is preferred when possible; open reduction with surgical hardware is necessary for some cases.
RICE (rest, ice, compression, and elevation) is beneficial for soft-tissue injuries, including those without musculoskeletal injuries. Rest prevents further inju

* Offers more in-depth medical considerations, including assessment for hemorrhagic shock, arterial/nerve injuries requiring specific diagnostics (arteriography, nerve conduction studies), and detailed immobilization techniques (splinting, casting). Specifically mentions the RICE protocol and different types of fracture reduction, along with rehabilitation considerations for various musculoskeletal issues.

### Testing if model answers non-medical query with RAG.

In [58]:
rag_qc_response = generate_rag_response(quality_control_query,k=1)
print(quality_control_query+"\n")
print(rag_qc_response)

Llama.generate: prefix-match hit


How to  clean a grease stain from a white shirt?

Answer: To clean a grease stain from a white shirt, you can use a cleansing agent such as dish soap or a detergent. Apply the soap or detergent directly onto the stain and work it in gently with your fingers. Rinse the area thoroughly with warm water and repeat if necessary. If the stain persists, you may need to use a stronger solvent like acetone or petroleum products, but be cautious as these can be drying and irritating to the skin. Alternatively, you could apply a petrolatum-based ointment or commercial waterless cleanser to help remove hardened tar or dried paint from the stain. Once the stain is removed, rinse the shirt thoroughly and wash it in warm water with detergent as usual.


* While still attempting to answer the non-medical query, the RAG model's response is notably more generic and shorter. It subtly tried to incorporate medical-adjacent language, advising to "avoid organic solvents... as they can be irritating and potentially cause contact dermatitis." This indicates the RAG system's attempt to adhere to its system prompt to "answer questions related to medicine" and "not answer any question that is not related to medicine" by filtering its response through a medical lens, even when the query is non-medical, demonstrating a stronger adherence to the prompt's role.

### Fine-tuning

Adjustments:

1. k=1, temperature = 0.25
2. k=3, temperature = 0
3. k=3, temperature = 0.25
4. k=1, temperature = 0.5
5. k=3, temperature = 0.5


#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [59]:
rag_finetuned_response_1 = generate_rag_response(query_1, k=1, temperature=0.25)
print(rag_finetuned_response_1)

Llama.generate: prefix-match hit


Answer: Sepsis is a life-threatening condition that requires prompt recognition and intervention. In a critical care unit, the following steps are typically taken to manage sepsis:
1. Early recognition: Septic patients should be identified as soon as possible based on clinical signs and laboratory results.
2. Fluid resuscitation: Patients with sepsis may be hypovolemic due to third spacing or capillary leak, so they require fluid replacement. Crystalloid solutions are usually the first choice for resuscitation.
3. Antibiotics: Broad-spectrum antibiotics should be administered as soon as possible based on the patient's clinical presentation and culture results.
4. Source control: The source of infection should be identified and treated if possible. This may involve surgical intervention or drainage procedures.
5. Vasopressors: Septic shock is characterized by hypotension that does not respond to fluid resuscitation. Vasopressors may be necessary to maintain adequate blood pressure.
6. C

* Again offers more specific and structured protocols, incorporating details from the medical manual such as specific types of vasopressors, and advanced treatment modalities (CRRT), or mentioning specific signs like tachycardia/bradycardia. This shows a deeper, context-aware understanding.

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [60]:
rag_finetuned_response_2 = generate_rag_response(query_2, k=3, temperature=0)
print(rag_finetuned_response_2)

Llama.generate: prefix-match hit


Answer: Appendicitis is a condition characterized by obstruction of the appendiceal lumen, typically due to lymphoid hyperplasia but occasionally caused by other factors such as fecalith or worms. The obstruction leads to distention, bacterial overgrowth, ischemia, and inflammation, which can result in necrosis, gangrene, perforation, and the formation of an appendiceal abscess if the perforation is contained by the omentum.
The classic symptoms of acute appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which later shifts to the right lower quadrant. Pain increases with cough and motion, and there may be direct and rebound tenderness at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right

* Provides a little more detail, including specific pain characteristics (epigastric/periumbilical shifting to right lower quadrant), precise medical signs (Rovsing, psoas, obturator, McBurney's point), and noted that classic findings are often absent. It detailed the medical consequences of non-treatment and the specific surgical procedures (laparotomy/laparoscopy), showing a more specialized and nuanced understanding from the context.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [61]:
rag_finetuned_response_3 = generate_rag_response(query_3, k=3, temperature=0.25)
print(rag_finetuned_response_3)

Llama.generate: prefix-match hit


Answer: Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects people with no obvious skin or systemic disorders. The scalp and beard are most frequently affected, but any hairy area may be involved. Hair loss may affect most or all of the body (alopecia universalis). Alopecia areata is thought to be an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers.
The treatment options for alopecia areata include:
1. Topical treatments: Corticosteroids, minoxidil, anthralin, or immunotherapy (diphencyprone or squaric acid dibutylester) can be applied directly to the affected area to stimulate hair growth and reduce inflammation.
2. Systemic treatments: In severe cases, oral corticosteroids or other immunosuppressive drugs may be prescribed to help suppress the immune system's attack on the hair follicles.
3. Hair transplant: Follicle transplant, scalp flaps, and alopecia reduction are surgical optio

* Expands on alopecia areata with specific corticosteroid types (topical, intralesional, systemic) and immunotherapies. Crucially, it also covered treatments for other hair loss conditions mentioned in the manual, such as traction alopecia, tinea capitis, and chemotherapy-induced hair loss, demonstrating a broader context-specific knowledge beyond just alopecia areata, leveraging the depth of the medical manual.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [62]:
rag_finetuned_response_4 = generate_rag_response(query_4, k=1, temperature=0.5)
print(rag_finetuned_response_4)

Llama.generate: prefix-match hit


Answer: Traumatic brain injury (TBI) can result in various degrees of neurological impairment, ranging from mild concussions to persistent vegetative states. In the case of a persistent vegetative state, where a person shows preservation of autonomic and motor reflexes, as well as normal sleep-wake cycles, but lacks consciousness or ability to interact with their environment, treatment options are limited. The Merck Manual indicates that few patients recover normal neurological function when this condition lasts for three months after the injury, and almost none do so after six months. While there is no cure for a persistent vegetative state, supportive care can help manage symptoms and improve quality of life. This may include measures to maintain body functions such as breathing, nutrition, hydration, and prevention of infections. Additionally, rehabilitation therapies like physical, occupational, and speech therapy can help improve function in areas not affected by the injury. Howev

* Lists specific injury outcomes like persistent vegetative state and brain death, detailing prognosis and the emphasis on supportive care and rehabilitation in such severe cases. It also mentiones specific neurological syndromes (agnosia, anosognosia), reflecting information likely found in the specialized manual rather than general knowledge.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [63]:
rag_finetuned_response_5 = generate_rag_response(query_5, k=3, temperature=0.5)
print(rag_finetuned_response_5)

Llama.generate: prefix-match hit


Answer: A fractured leg is a serious injury that requires prompt medical attention. The first step in managing this injury is to ensure that the person's airway is clear and they are breathing normally. If there are signs of shock, such as pale skin, rapid heartbeat, or low blood pressure, immediate steps should be taken to stabilize their condition, including elevating their legs above heart level and administering fluids if necessary.
For suspected arterial injuries, arteriography may be required for proper diagnosis and treatment. Nerve conduction studies may also be indicated for nerve injuries.
Initial treatment for most fractures involves immobilization to prevent further injury and decrease pain. Splinting with a nonrigid or noncircumferential device is often used to hold the injured limb in place until definitive treatment can be provided. Definitive treatment, such as reduction, may require analgesia or sedation and could involve closed or open reduction depending on the sever

* Offers more in-depth medical considerations, including assessment for hemorrhagic shock, arterial/nerve injuries requiring specific diagnostics (arteriography, nerve conduction studies), and detailed immobilization techniques (splinting, casting). Mentions the RICE protocol and different types of fracture reduction, along with rehabilitation considerations for various musculoskeletal issues, showing a more clinical and detailed approach from the manual.

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [64]:
groundedness_rater_system_message = "You are an expert at evaluating the groundedness of a response. You will be provided with a question, a context, and an answer. Your task is to rate the answer's groundedness on a scale of 1 to 5, where 1 means the answer is not at all grounded in the context and 5 means the answer is entirely grounded in the context. Provide a concise explanation for your rating."

In [65]:
relevance_rater_system_message = "You are an expert at evaluating the relevance of a response. You will be provided with a question, a context, and an answer. Your task is to rate the answer's relevance on a scale of 1 to 5, where 1 means the answer is not at all relevant to the question and 5 means the answer is entirely relevant to the question. Provide a concise explanation for your rating."

In [66]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [67]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=512,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [68]:
ground,rel = generate_ground_relevance_response(user_input=query_1, max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Answer rating: 5

The answer is entirely grounded in the context provided. The context discusses various aspects of critical care medicine, including patient monitoring, testing, and treatment of sepsis or septic shock. The answer outlines the steps for managing a suspected case of sepsis or septic shock in a critical care unit, which aligns with the information presented in the context.

 Answer rating: 5

The answer is entirely relevant to the question as it provides a clear and comprehensive protocol for managing sepsis in a critical care unit. The steps outlined in the answer align with the information provided in the context, including obtaining cultures, initiating empiric antibiotic therapy, fluid resuscitation, monitoring vital signs, adjusting antibiotics based on culture results, surgical intervention if necessary, removing internal devices, and providing supportive care.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [69]:
ground,rel = generate_ground_relevance_response(user_input=query_2, max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Answer rating: 5

The answer is entirely grounded in the context provided by the text. The answer accurately summarizes the etiology, symptoms, signs, diagnosis, and treatment of appendicitis as described in the context. The answer also acknowledges the variations in symptoms and signs that may occur and mentions the role of imaging studies in diagnosing atypical cases.

 I would rate the answer as a 5 for relevance to the question. The user's response accurately summarizes the etiology, symptoms, signs, diagnosis, and treatment of appendicitis, addressing all aspects of the original question. The context provided in the text also supports the accuracy of the information given in the answer.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [70]:
ground,rel = generate_ground_relevance_response(user_input=query_3, max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Answer: 5

The answer is entirely grounded in the context as it accurately identifies alopecia areata as a common cause of sudden patchy hair loss and discusses various treatment options, including topical corticosteroids, minoxidil, intralesional corticosteroid injections, systemic corticosteroids or other immunosuppressive drugs, hair transplant, elimination of physical traction or stress to the scalp, and treatment of underlying disorders. The answer also explains that alopecia areata is an autoimmune disorder affecting genetically susceptible individuals exposed to unclear environmental triggers, which aligns with the information provided in the context.

 Answer: I would rate this answer as a 5 for relevance. The user's question asked about effective treatments and possible causes of sudden patchy hair loss, specifically alopecia areata. The answer provided detailed information on both aspects of the question, explaining that alopecia areata is an autoimmune disorder causing sudd

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [71]:
ground,rel = generate_ground_relevance_response(user_input=query_4, max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Rating: 5

The answer is entirely grounded in the context as it summarizes the information provided in the context regarding the diagnosis, prognosis, and treatment options for brain injuries, including persistent vegetative state, brain death, and specific syndromes such as agnosia and anosognosia. The answer accurately reflects the text by emphasizing the importance of the extent and location of the damage, age, and overall health in determining the treatment and prognosis for individuals with brain injuries.

 Rating: 5

The answer is entirely relevant to the question as it discusses various treatments and prognosis for a person with a brain injury based on the extent and location of the damage, individual's age, and overall health. It also mentions specific syndromes resulting from brain injury such as agnosia and anosognosia, and their diagnosis and treatment.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [72]:
ground,rel = generate_ground_relevance_response(user_input=query_5, max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer as a 5 for groundedness in the context. The answer accurately summarizes the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, based on the provided context from The Merck Manual of Diagnosis & Therapy. It covers various aspects such as initial assessment and treatment for life-threatening injuries, nerve injuries, arterial injuries, splinting, definitive treatment, RICE therapy, immobilization, and patient instructions.

 I would rate the answer as a 5 for relevance. The response directly addresses the question by discussing necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, including immobilization techniques, RICE therapy, and potential surgical interventions for arterial or nerve injuries. The context provided in the question is also referenced throughout the answer to provide additional context and detail.


* The consistent high scores for ‘groundedness’ and ‘relevance’ (5/5) in the evaluation demonstrate that the AI’s responses are directly supported by the medical manual. This builds confidence in the system’s output, crucial for medical applications where accuracy is paramount.

## Final test: answering common questions with fine-tuned RAG.

In [73]:
print(common_question_1+"\n")

print(generate_rag_response(common_question_1, k=3, temperature = 0.5))

What are the common symptoms and treatments for pulmonary embolism?



Llama.generate: prefix-match hit


Answer: Pulmonary embolism (PE) is a condition characterized by the occlusion of one or more pulmonary arteries by thrombi that originate elsewhere, typically in the lower extremities or pelvis. Common risk factors include conditions that impair venous return, endothelial injury or dysfunction, and underlying hypercoagulable states. Symptoms are nonspecific and may include dyspnea, pleuritic chest pain, cough, syncope, or cardiorespiratory arrest. Diagnosis is based on imaging studies such as CT angiogram, ventilation/perfusion scan, or pulmonary arteriogram. Treatment includes anticoagulants and, in some cases, thrombolytics or surgical removal of the clots. Preventive measures include anticoagulants and inferior vena caval filters. PE affects about 350,000 people yearly and causes up to 85,000 deaths per year. The condition can lead to acute right ventricular failure, shock, or sudden death if left untreated. Small emboli may have no acute physiologic effects and resolve on their own

In [74]:
print(common_question_2+"\n")

print(generate_rag_response(common_question_2, k=3, temperature = 0.5))

What treatment options are available for managing hypertension?



Llama.generate: prefix-match hit


Answer: The treatment for hypertension involves both lifestyle modifications and medications. Lifestyle changes include reducing sodium intake, maintaining a healthy weight through regular exercise, limiting alcohol consumption, and avoiding tobacco use. Medications are usually prescribed when lifestyle modifications alone are not enough to control blood pressure (BP). Commonly used antihypertensive drugs include diuretics, beta-blockers, angiotensin converting enzyme (ACE) inhibitors, angiotensin II receptor blockers (ARBs), and calcium channel blockers. ACE inhibitors and ARBs are the first-line therapies due to their ability to reduce BP and proteinuria, slowing down the progression of kidney damage in patients with chronic kidney disease. Diuretics are often used in combination with other medications to help reach target BP levels. Nondihydropyridine calcium channel blockers can also be used as they have antiproteinuric and renal protective effects. Treatment should be started as s

In [75]:
print(common_question_3+"\n")

print(generate_rag_response(common_question_3, k=3, temperature = 0.5))

Can you provide the trade names of medications used for treating hypertension?



Llama.generate: prefix-match hit


Answer: Yes, some common trade names for medications used to treat hypertension include hydrochlorothiazide (Diuril, Esidrix), lisinopril (Prinivil, Zestril), metformin (Glucophage), and nifedipine (Procardia). However, it's important to note that there are many other trade names for each class of medication used for hypertension treatment. It's always best to consult a healthcare professional or pharmacist for the most accurate information regarding specific medications and their brand names.


In [76]:
print(common_question_4+"\n")

print(generate_rag_response(common_question_4, k=3, temperature = 0.5))

What are the first-line options and alternatives for managing rheumatoid arthritis?



Llama.generate: prefix-match hit


Answer: The first-line options for managing rheumatoid arthritis (RA) include disease-modifying antirheumatic drugs (DMARDs), such as methotrexate, leflunomide, or sulfasalazine. These medications help control symptoms and slow disease progression. DMARDS may be used alone or in combination with other medications, such as nonsteroidal anti-inflammatory drugs (NSAIDs) or corticosteroids, to manage pain and inflammation.
Alternative options for managing RA include physical measures, such as joint immobilization with splints or slings, heat therapy, and cold therapy. Surgery may also be considered in cases of severe damage or disability.
It is important to note that the specific treatment approach for RA depends on various factors, including the severity and duration of the disease, individual patient characteristics, and response to previous treatments. Close monitoring and follow-up are essential to ensure effective management of RA and to minimize potential side effects of medications.

In [77]:
print(common_question_5+"\n")

print(generate_rag_response(common_question_5, k=3, temperature = 0.5))

What are the diagnostic steps for suspected endocrine disorders?



Llama.generate: prefix-match hit


Answer: The diagnosis of suspected endocrine disorders involves a thorough review of systems to identify symptoms suggestive of possible causes, a detailed past medical history, and a comprehensive physical examination. During the physical examination, vital signs, skin, and general appearance are assessed. In cases of gynecomastia, the neck is examined for goiter, the abdomen for ascites, venous distention, and suspected adrenal masses, and secondary sexual characteristics are evaluated. The breasts are examined while patients are recumbent with their hands behind their head to assess for lumps, consistency, fixation to underlying tissues, and skin changes. Red flags such as localized or eccentric breast swelling, symptoms or signs of hypogonadism, hyperthyroidism, testicular masses, or recent onset of painful, tender gynecomastia in an adult warrant further investigation. Diagnosis is confirmed by measuring hormone levels and sometimes autoantibody titers. Lifelong follow-up is neces

## Actionable Insights and Business Recommendations

### Actionable Insights

*   LLM-only responses were generally comprehensive but generic, relying on the model's pre-trained knowledge. They offered good general information but lacked the specific, nuanced details that a specialized medical professional would require.

*   LLM with System Prompt responses showed a slight improvement in structure and adherence to constraints (like summarization), but still primarily drew from general knowledge and occasionally failed to strictly adhere to the medical-only constraint (as seen with the non-medical query).

*   RAG-enhanced responses (both initial and fine-tuned) demonstrated a significant improvement in groundedness and specificity. By leveraging the provided medical manual as context, these responses were able to offer more precise diagnostic criteria, detailed protocols, and specific treatment options, better reflecting the source material.

*   Hosting a local or open LLM, plus using RAG architecture allows for easy updates and expansion of the knowledge base by simply adding new or updated medical texts. This ensures the AI solution remains current with the latest medical advancements and guidelines, and with a lower cost than using external closed LLMs.

### Business Recommendations

*   **Implement as a Support Tool:** Deploying the model with RAG enhancement as a tool for physicians, nurses, and medical students, could help in answering complex clinical questions, confirming diagnoses, and outlining treatment plans supported directly by data from a trusted source.

*   **Continuous Knowledge Base Expansion:** Regularly update the vector database with the latest editions of medical manuals, research papers, and clinical guidelines to ensure the AI's information remains current and comprehensive. Consider including specialized manuals for different medical departments based on specific needs (e.g., emergency medicine, oncology, cardiology)

*   **Training Programs:** Develop training programs for healthcare staff on how to effectively use the model and create better prompts. By highlighting its benefits in improving patient outcomes and reducing cognitive load, it will help to drive adoption.

<font size=6 color='blue'>Power Ahead</font>
___

Disclaimer: this notebook uses some of the code and structure from: Low_Code_Medical_Assistant_Notebook.ipynb, Week_4_Guided_Hands_on_Notebook_RAG_Notebook.ipynb and Hands_on_Prompt_Engineering_Notebook.ipynb; and some AI generated recommendations (for code) from Goggle Colab.